# Structured text and line media - Python

All 8 Python examples from [docs/text.md](https://platob.github.io/yggdryl/text/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

## Text media and Arrow batches

In [ ]:
import pathlib
import shutil
import tempfile

import pytest

from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp(prefix="yggdryl-doc-"))
(root / "app.log").write_text("first event\nsecond event\n")

table = IOBase(root / "app.log").read_arrow_reader().read_all()
assert table.num_rows == 2
assert table.column("message").to_pylist() == ["first event", "second event"]

target = IOBase(root / "copy.log")
target.overwrite_arrow_table(table)
target.append_arrow_table(table)
assert (root / "copy.log").read_text() == (
    "first event\nsecond event\nfirst event\nsecond event\n"
)

merging = target.record_options()
merging.merge_by_names = ["message"]
with pytest.raises(ValueError, match="row identity"):
    target.merge_arrow_table(table, options=merging)

shutil.rmtree(root)

### Line iteration with `Text`

In [ ]:
import pathlib
import tempfile

from yggdryl import IOBase

with tempfile.TemporaryDirectory() as directory:
    source = pathlib.Path(directory) / "app.log"
    source.write_bytes(
        b"2026-08-01 [ERROR] failed\n  detail\n"
        b"2026-08-01 [INFO] ready\n"
    )
    records = list(
        IOBase(source).read_lines(r"^\d{4}-\d{2}-\d{2} \[[A-Z]+\]")
    )

    assert len(records) == 2
    assert "detail" in records[0]

## Raw shared-Scalar access

In [ ]:
from yggdryl import Scalar, json

quote = json.loads('{"symbol":"AAPL","price":12.5}', cls=Scalar)

assert quote["symbol"].as_utf8() == "AAPL"
assert quote.path("price").kind == "f64"
assert quote.set("venue", "XNAS").get("venue").as_utf8() == "XNAS"
assert quote.as_py() == {"price": 12.5, "symbol": "AAPL"}

### Typed `Scalar` families

In [ ]:
from yggdryl import Scalar

assert (Scalar.from_py(40) + 2).as_py() == 42
assert Scalar.decimal(1, 0).divide(Scalar.decimal(2, 0)) == Scalar.decimal(5, 1)

## Field-directed parsing

In [ ]:
from decimal import Decimal

from yggdryl import Field, Scalar, json

amount = Field("amount", "decimal128(8, 2)", nullable=False)
value = json.loads('"12.50"', field=amount, cls=Scalar)

assert value.kind == "d128"
assert value.unscaled == 1_250
assert json.loads('"12.50"', field=amount) == Decimal("12.50")

## Raw document codecs

In [ ]:
from yggdryl import Scalar, codec

value = codec.from_io('{"id":1}', cls=Scalar)

assert isinstance(value, Scalar)
assert value["id"].kind == "u64"
assert codec.into_io(value, format="json", utf8=True) == '{"id":1}'

## Formatting

In [ ]:
from yggdryl import json

pretty = json.dumps({"id": 1}, indent=2)
compact = json.dumps({"id": 1}, indent=None)

assert pretty == b'{\n  "id": 1\n}'
assert compact == b'{"id":1}'

## Placeholders

In [ ]:
from yggdryl import yaml

document = 'host: "{{ HOST }}"\nport: "{{ PORT | default(8080) }}"\n'
value = yaml.loads(document, placeholders={"HOST": "db.internal"})

assert value == {"host": "db.internal", "port": 8080}